# V18-C11: LIANA Tissue-Separated Cell-Cell Interaction Analysis
## Internal Verification — Not for manuscript inclusion (unless warranted)

**Date:** 2026-03-05
**Purpose:**
1. Verify V18 paracrine interaction hypotheses (TGFB1→TGFBR2, LGALS9→HAVCR2, etc.)
2. Compare Liver vs Blood interaction landscapes — does tissue separation matter for interactions too?
3. Compare with Zhang et al. CSOmap results — concordance or discrepancy?
4. Discover unexpected interactions not captured by single-gene analysis

**Philosophy:** This is about understanding our data deeply, not about adding a figure to the paper.

**Analysis unit:** Per disease group × per tissue. NOT combined.

---
## Cell 1: Install LIANA & Dependencies

In [ ]:
# ============================================================
# Cell 1: Install liana-py
# ============================================================
!pip install liana --quiet
!pip install omnipath --quiet

import liana
print(f'LIANA version: {liana.__version__}')

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.dpi'] = 150
matplotlib.rcParams['figure.facecolor'] = 'white'

from google.colab import drive
drive.mount('/content/drive')

import os
RESULTS_DIR = '/content/drive/MyDrive/ITLAS/results/version18-analysis/C11_LIANA/'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Setup complete.')

---
## Cell 2: Load Data & Set Column Names (from C10)

In [ ]:
# ============================================================
# Cell 2: Load data — reuse C10 column conventions
# ============================================================
DATA_PATH = '/content/drive/MyDrive/ITLAS/data/processed/GSE182159_gut2021_annotated.h5ad'
print('Loading h5ad...')
adata = sc.read_h5ad(DATA_PATH)
print(f'Loaded: {adata.shape[0]} cells x {adata.shape[1]} genes')

# Column names (validated in C10)
COL_DISEASE = 'Stage'
COL_LINEAGE = 'major_lineage'
COL_TISSUE = 'tissue'
COL_SUBCLUSTER = 'gut2021_subcluster_v2'

# Create donor_id from sample (validated in C10)
adata.obs['donor_id'] = adata.obs['sample'].str.split('_').str[1]
COL_DONOR = 'donor_id'

print(f'Donors: {adata.obs[COL_DONOR].nunique()}')
print(f'Subclusters: {adata.obs[COL_SUBCLUSTER].nunique()}')
print(f'Lineages: {sorted(adata.obs[COL_LINEAGE].unique())}')
print(f'Tissues: {sorted(adata.obs[COL_TISSUE].unique())}')
print(f'Disease groups: {sorted(adata.obs[COL_DISEASE].unique())}')

---
## Cell 3: Define V18 Hypothesized Interactions
These are the ligand-receptor pairs implied by our 6-pattern model.
LIANA will tell us if they are statistically enriched.

In [ ]:
# ============================================================
# Cell 3: V18 hypothesized interactions to verify
# ============================================================

V18_HYPOTHESES = {
    # Pattern 1: Myeloid paracrine suppression
    'P1_TGFB1_TGFBR2': {'ligand': 'TGFB1', 'receptor': 'TGFBR2',
                          'source': 'Myeloid', 'target': ['CD4_T', 'CD8_T', 'NK', 'B'],
                          'expected': 'IT > NL'},
    'P1_TGFB1_TGFBR1': {'ligand': 'TGFB1', 'receptor': 'TGFBR1',
                          'source': 'Myeloid', 'target': ['CD4_T', 'CD8_T', 'NK'],
                          'expected': 'IT > NL'},
    'P1_LGALS9_HAVCR2': {'ligand': 'LGALS9', 'receptor': 'HAVCR2',
                          'source': 'Myeloid', 'target': ['CD4_T', 'CD8_T', 'NK'],
                          'expected': 'IT > NL (Tim-3 axis)'},
    'P1_HLA_CD4':       {'ligand': 'HLA-DRA', 'receptor': 'CD4',
                          'source': 'Myeloid', 'target': ['CD4_T'],
                          'expected': 'IT > NL (Ag presentation)'},

    # Pattern 5: Liver exhaustion
    'P5_LGALS9_HAVCR2_liver': {'ligand': 'LGALS9', 'receptor': 'HAVCR2',
                                'source': 'Myeloid', 'target': ['CD4_T', 'CD8_T'],
                                'expected': 'Liver IT >> Blood IT'},

    # Additional interactions from Zhang et al.
    'Zhang_FCGR3A_Tex': {'ligand': 'FCGR3A', 'receptor': None,
                          'source': 'Myeloid', 'target': ['CD8_T'],
                          'expected': 'Zhang reported this in IA'},

    # B cell interactions
    'B_IL2RA':          {'ligand': 'IL2', 'receptor': 'IL2RA',
                          'source': ['CD4_T', 'CD8_T'], 'target': ['B'],
                          'expected': 'IT > NL (B activation)'},
}

print(f'Defined {len(V18_HYPOTHESES)} hypothesized interactions to verify')
for name, hyp in V18_HYPOTHESES.items():
    print(f'  {name}: {hyp["ligand"]} -> {hyp.get("receptor", "?")} '
          f'({hyp["source"]} -> {hyp["target"]})')

---
## Cell 4: Run LIANA — Per Disease Group × Per Tissue
This is the core analysis. We run LIANA separately for each condition.

In [ ]:
# ============================================================
# Cell 4: Run LIANA for each tissue × disease group
# Uses subcluster (59 types) as cell identity for interaction
# ============================================================
from liana.method import singlecellsignalr, connectome, cellphonedb, natmi, logfc, cellchat, geometric_mean
# We use the multi-method consensus approach
from liana import multi as liana_multi

TISSUES = ['Liver', 'Blood']
DISEASE_GROUPS = ['NL', 'IT', 'IA', 'AR', 'CR']

# Use major_lineage as groupby for cleaner interpretation
# (subcluster gives more granularity but harder to interpret)
GROUPBY = COL_LINEAGE  # or COL_SUBCLUSTER for granular analysis

liana_results = {}

for tissue in TISSUES:
    for disease in DISEASE_GROUPS:
        key = f'{tissue}_{disease}'
        print(f'\n=== Running LIANA: {key} ===')
        
        # Subset
        mask = (adata.obs[COL_TISSUE] == tissue) & (adata.obs[COL_DISEASE] == disease)
        sub = adata[mask].copy()
        
        n_cells = sub.shape[0]
        n_donors = sub.obs[COL_DONOR].nunique()
        lineages = sub.obs[GROUPBY].value_counts()
        print(f'  Cells: {n_cells}, Donors: {n_donors}')
        print(f'  Lineages: {dict(lineages)}')
        
        # Skip if too few cells
        if n_cells < 100:
            print(f'  SKIP: too few cells ({n_cells})')
            continue
        
        # Filter lineages with < 10 cells
        valid_lineages = lineages[lineages >= 10].index.tolist()
        sub = sub[sub.obs[GROUPBY].isin(valid_lineages)].copy()
        print(f'  Valid lineages (>=10 cells): {valid_lineages}')
        
        # Ensure raw counts or log-normalized in .X
        # LIANA expects log-normalized by default
        # The h5ad should already be log-normalized
        
        try:
            # Run LIANA with multiple methods
            liana.mt.rank_aggregate(
                sub,
                groupby=GROUPBY,
                resource_name='consensus',  # OmniPath consensus resource
                expr_prop=0.1,  # min proportion of cells expressing gene
                verbose=True,
                use_raw=False,  # use .X (log-normalized)
            )
            
            # Extract results
            result = sub.uns['liana_res'].copy()
            result['tissue'] = tissue
            result['disease'] = disease
            liana_results[key] = result
            
            print(f'  ✅ {len(result)} interactions found')
            # Show top 10
            top = result.nsmallest(10, 'magnitude_rank')
            print(top[['source', 'target', 'ligand_complex', 'receptor_complex', 
                       'magnitude_rank', 'specificity_rank']].to_string())
            
        except Exception as e:
            print(f'  ❌ ERROR: {e}')
            continue

print(f'\n==> LIANA completed for {len(liana_results)} conditions')

---
## Cell 5: Verify V18 Hypothesized Interactions

In [ ]:
# ============================================================
# Cell 5: Check V18 hypothesized interactions in LIANA results
# ============================================================

def find_interaction(liana_df, ligand, receptor=None, source=None, target=None):
    """Find specific ligand-receptor interaction in LIANA results."""
    mask = pd.Series([True] * len(liana_df))
    
    if ligand:
        mask &= liana_df['ligand_complex'].str.contains(ligand, case=False, na=False)
    if receptor:
        mask &= liana_df['receptor_complex'].str.contains(receptor, case=False, na=False)
    if source:
        if isinstance(source, list):
            mask &= liana_df['source'].isin(source)
        else:
            mask &= liana_df['source'] == source
    if target:
        if isinstance(target, list):
            mask &= liana_df['target'].isin(target)
        else:
            mask &= liana_df['target'] == target
    
    return liana_df[mask]

print('=' * 80)
print('  V18 HYPOTHESIZED INTERACTIONS — LIANA VERIFICATION')
print('=' * 80)

verification_records = []

for hyp_name, hyp in V18_HYPOTHESES.items():
    print(f'\n--- {hyp_name} ---')
    print(f'    Expected: {hyp["expected"]}')
    
    for tissue in TISSUES:
        for disease in ['NL', 'IT', 'IA']:
            key = f'{tissue}_{disease}'
            if key not in liana_results:
                continue
            
            matches = find_interaction(
                liana_results[key],
                ligand=hyp['ligand'],
                receptor=hyp.get('receptor'),
                source=hyp.get('source') if not isinstance(hyp.get('source'), list) else None,
                target=None  # search broadly first
            )
            
            if len(matches) > 0:
                best = matches.nsmallest(1, 'magnitude_rank').iloc[0]
                rank = best['magnitude_rank']
                spec = best['specificity_rank']
                src = best['source']
                tgt = best['target']
                lig = best['ligand_complex']
                rec = best['receptor_complex']
                
                status = '✅ TOP' if rank < 0.1 else ('⚠️ MID' if rank < 0.3 else '❌ WEAK')
                print(f'    {tissue}/{disease}: {src}→{tgt} {lig}|{rec} '
                      f'mag_rank={rank:.3f} spec_rank={spec:.3f} {status}')
                
                verification_records.append({
                    'hypothesis': hyp_name,
                    'tissue': tissue, 'disease': disease,
                    'source': src, 'target': tgt,
                    'ligand': lig, 'receptor': rec,
                    'magnitude_rank': rank,
                    'specificity_rank': spec,
                    'status': status
                })
            else:
                print(f'    {tissue}/{disease}: NOT FOUND')
                verification_records.append({
                    'hypothesis': hyp_name,
                    'tissue': tissue, 'disease': disease,
                    'source': '', 'target': '',
                    'ligand': hyp['ligand'], 'receptor': hyp.get('receptor', ''),
                    'magnitude_rank': 1.0, 'specificity_rank': 1.0,
                    'status': '❌ ABSENT'
                })

verif_df = pd.DataFrame(verification_records)
verif_df.to_csv(f'{RESULTS_DIR}C11_V18_hypothesis_verification.csv', index=False)
print(f'\nSaved verification results to {RESULTS_DIR}')

---
## Cell 6: Liver vs Blood Interaction Landscape Comparison
**Core question: Does tissue separation matter for cell-cell interactions?**

In [ ]:
# ============================================================
# Cell 6: Compare Liver vs Blood interaction landscapes
# For each disease group: what are the top interactions in each tissue?
# How much overlap is there?
# ============================================================

print('=' * 80)
print('  LIVER vs BLOOD: INTERACTION LANDSCAPE COMPARISON')
print('=' * 80)

for disease in ['NL', 'IT', 'IA']:
    liver_key = f'Liver_{disease}'
    blood_key = f'Blood_{disease}'
    
    if liver_key not in liana_results or blood_key not in liana_results:
        print(f'\n{disease}: Missing data for one tissue, skip')
        continue
    
    liver_df = liana_results[liver_key]
    blood_df = liana_results[blood_key]
    
    # Top 50 interactions by magnitude rank
    liver_top50 = set(liver_df.nsmallest(50, 'magnitude_rank').apply(
        lambda r: f"{r['source']}|{r['ligand_complex']}|{r['receptor_complex']}|{r['target']}", axis=1))
    blood_top50 = set(blood_df.nsmallest(50, 'magnitude_rank').apply(
        lambda r: f"{r['source']}|{r['ligand_complex']}|{r['receptor_complex']}|{r['target']}", axis=1))
    
    overlap = liver_top50 & blood_top50
    liver_only = liver_top50 - blood_top50
    blood_only = blood_top50 - liver_top50
    
    print(f'\n=== {disease} ===')
    print(f'  Top 50 Liver interactions: {len(liver_top50)}')
    print(f'  Top 50 Blood interactions: {len(blood_top50)}')
    print(f'  Overlap: {len(overlap)} ({len(overlap)/50*100:.0f}%)')
    print(f'  Liver-only: {len(liver_only)}')
    print(f'  Blood-only: {len(blood_only)}')
    
    if len(liver_only) > 0:
        print(f'\n  Top 5 LIVER-ONLY interactions:')
        for interaction in sorted(liver_only)[:5]:
            parts = interaction.split('|')
            print(f'    {parts[0]} → {parts[3]}: {parts[1]} | {parts[2]}')
    
    if len(blood_only) > 0:
        print(f'\n  Top 5 BLOOD-ONLY interactions:')
        for interaction in sorted(blood_only)[:5]:
            parts = interaction.split('|')
            print(f'    {parts[0]} → {parts[3]}: {parts[1]} | {parts[2]}')

---
## Cell 7: IT vs NL — Which Interactions Emerge or Disappear?

In [ ]:
# ============================================================
# Cell 7: IT vs NL differential interactions
# For each tissue: which interactions are stronger in IT than NL?
# ============================================================

def compare_interactions(df1, df2, label1='NL', label2='IT', top_n=100):
    """Compare interaction ranks between two conditions."""
    # Create interaction key
    def make_key(df):
        return df.apply(
            lambda r: f"{r['source']}|{r['ligand_complex']}|{r['receptor_complex']}|{r['target']}",
            axis=1
        )
    
    df1 = df1.copy()
    df2 = df2.copy()
    df1['key'] = make_key(df1)
    df2['key'] = make_key(df2)
    
    # Merge on interaction key
    merged = df1[['key', 'source', 'target', 'ligand_complex', 'receptor_complex',
                  'magnitude_rank']].merge(
        df2[['key', 'magnitude_rank']],
        on='key', how='outer', suffixes=(f'_{label1}', f'_{label2}')
    )
    
    # Fill missing with 1.0 (worst rank = absent)
    merged[f'magnitude_rank_{label1}'] = merged[f'magnitude_rank_{label1}'].fillna(1.0)
    merged[f'magnitude_rank_{label2}'] = merged[f'magnitude_rank_{label2}'].fillna(1.0)
    
    # Delta: negative = stronger in label2 (IT)
    merged['delta_rank'] = merged[f'magnitude_rank_{label2}'] - merged[f'magnitude_rank_{label1}']
    
    return merged

print('=' * 80)
print('  IT vs NL: DIFFERENTIAL INTERACTIONS')
print('=' * 80)

for tissue in TISSUES:
    nl_key = f'{tissue}_NL'
    it_key = f'{tissue}_IT'
    
    if nl_key not in liana_results or it_key not in liana_results:
        continue
    
    diff = compare_interactions(liana_results[nl_key], liana_results[it_key])
    
    # IT-enriched: much stronger in IT (large negative delta)
    it_enriched = diff.nsmallest(15, 'delta_rank')
    # IT-depleted: much weaker in IT (large positive delta)
    it_depleted = diff.nlargest(15, 'delta_rank')
    
    print(f'\n=== {tissue}: IT-ENRICHED interactions (stronger in IT than NL) ===')
    for _, row in it_enriched.iterrows():
        parts = row['key'].split('|') if pd.notna(row.get('key')) else ['?']*4
        print(f'  {parts[0]}→{parts[3]}: {parts[1]}|{parts[2]} '
              f'NL_rank={row["magnitude_rank_NL"]:.3f} IT_rank={row["magnitude_rank_IT"]:.3f} '
              f'delta={row["delta_rank"]:+.3f}')
    
    print(f'\n=== {tissue}: IT-DEPLETED interactions (weaker in IT than NL) ===')
    for _, row in it_depleted.iterrows():
        parts = row['key'].split('|') if pd.notna(row.get('key')) else ['?']*4
        print(f'  {parts[0]}→{parts[3]}: {parts[1]}|{parts[2]} '
              f'NL_rank={row["magnitude_rank_NL"]:.3f} IT_rank={row["magnitude_rank_IT"]:.3f} '
              f'delta={row["delta_rank"]:+.3f}')
    
    # Save
    diff.to_csv(f'{RESULTS_DIR}C11_differential_{tissue}_IT_vs_NL.csv', index=False)

---
## Cell 8: Granular Analysis with Subclusters
Repeat for IT using subcluster-level grouping for finer resolution.

In [ ]:
# ============================================================
# Cell 8: Subcluster-level LIANA for IT (finer resolution)
# Only for IT phase in both tissues — most relevant for our study
# ============================================================

for tissue in TISSUES:
    print(f'\n=== Subcluster-level LIANA: {tissue} IT ===')
    
    mask = (adata.obs[COL_TISSUE] == tissue) & (adata.obs[COL_DISEASE] == 'IT')
    sub = adata[mask].copy()
    
    # Filter subclusters with >= 20 cells
    sc_counts = sub.obs[COL_SUBCLUSTER].value_counts()
    valid_sc = sc_counts[sc_counts >= 20].index.tolist()
    sub = sub[sub.obs[COL_SUBCLUSTER].isin(valid_sc)].copy()
    
    print(f'  Cells: {sub.shape[0]}, Subclusters (>=20 cells): {len(valid_sc)}')
    
    try:
        liana.mt.rank_aggregate(
            sub,
            groupby=COL_SUBCLUSTER,
            resource_name='consensus',
            expr_prop=0.1,
            verbose=True,
            use_raw=False,
        )
        
        result = sub.uns['liana_res'].copy()
        result.to_csv(f'{RESULTS_DIR}C11_subcluster_{tissue}_IT.csv', index=False)
        
        print(f'  ✅ {len(result)} interactions')
        
        # Check myeloid → T cell interactions specifically
        myeloid_sc = [s for s in valid_sc if 'mono' in s.lower() or 'dc' in s.lower() or 'cdc' in s.lower()]
        tcell_sc = [s for s in valid_sc if 'cd4' in s.lower() or 'cd8' in s.lower()]
        
        if myeloid_sc and tcell_sc:
            myeloid_to_tcell = result[
                result['source'].isin(myeloid_sc) & result['target'].isin(tcell_sc)
            ].nsmallest(20, 'magnitude_rank')
            
            print(f'\n  Top 20 Myeloid→T cell interactions:')
            print(myeloid_to_tcell[['source', 'target', 'ligand_complex', 'receptor_complex',
                                    'magnitude_rank']].to_string())
    
    except Exception as e:
        print(f'  ❌ ERROR: {e}')

---
## Cell 9: Summary Report

In [ ]:
# ============================================================
# Cell 9: Summary
# ============================================================

print('=' * 80)
print('  C11 LIANA TISSUE-SEPARATED ANALYSIS — SUMMARY')
print('=' * 80)

print('\n[1] CONDITIONS ANALYZED')
for key in sorted(liana_results.keys()):
    print(f'  {key}: {len(liana_results[key])} interactions')

print('\n[2] V18 HYPOTHESIS VERIFICATION')
if len(verif_df) > 0:
    for hyp_name in V18_HYPOTHESES.keys():
        hyp_rows = verif_df[verif_df['hypothesis'] == hyp_name]
        statuses = hyp_rows['status'].value_counts().to_dict()
        print(f'  {hyp_name}: {statuses}')

print('\n[3] TISSUE DISCREPANCY IN INTERACTIONS')
print('  (See Cell 6 output for Liver vs Blood overlap analysis)')
print('  Key question: If overlap < 50%, tissue separation is critical for interactions too.')

print('\n[4] KEY FINDINGS FOR INTERNAL REFERENCE')
print('  - Does LIANA confirm TGFB1-TGFBR2 Myeloid→T cell interaction in IT?')
print('  - Does LGALS9-HAVCR2 (Tim-3) axis appear in Liver but not Blood?')
print('  - Are there unexpected interactions we missed in gene-level analysis?')
print('  - Does the interaction landscape differ between Liver and Blood?')

print('\n[5] FILES GENERATED')
for f in sorted(os.listdir(RESULTS_DIR)):
    if f.startswith('C11_'):
        size = os.path.getsize(os.path.join(RESULTS_DIR, f))
        print(f'  {f} ({size/1024:.1f} KB)')

print('\n==> C11 LIANA Analysis Complete.')
print('\nNote: These results are for internal verification.')
print('Include in manuscript only if they reveal something')
print('that fundamentally changes or strengthens our conclusions.')